In [16]:
import pandas as pd

URL = "https://github.com/nflverse/nflverse-data/releases/download/draft_picks/draft_picks.csv"

draft = pd.read_csv(URL)
print("shape:", draft.shape)
print("seasons:", draft["season"].min(), "->", draft["season"].max())
print(list(draft.columns))
draft.head()

shape: (12927, 36)
seasons: 1980 -> 2026
['season', 'round', 'pick', 'team', 'gsis_id', 'pfr_player_id', 'cfb_player_id', 'pfr_player_name', 'hof', 'position', 'category', 'side', 'college', 'age', 'to', 'allpro', 'probowls', 'seasons_started', 'w_av', 'car_av', 'dr_av', 'games', 'pass_completions', 'pass_attempts', 'pass_yards', 'pass_tds', 'pass_ints', 'rush_atts', 'rush_yards', 'rush_tds', 'receptions', 'rec_yards', 'rec_tds', 'def_solo_tackles', 'def_ints', 'def_sacks']


,season,round,pick,team,gsis_id,pfr_player_id,cfb_player_id,pfr_player_name,hof,position,...,pass_ints,rush_atts,rush_yards,rush_tds,receptions,rec_yards,rec_tds,def_solo_tackles,def_ints,def_sacks
0,1980,1,1,DET,SIM659150,SimsBi00,billy-sims-1,Billy Sims,False,RB,...,0.0,1131.0,5106.0,42.0,186.0,2072.0,5.0,NaN,NaN,NaN
1,1980,1,2,NYJ,JON491656,JoneLa00,lam-jones-1,Lam Jones,False,WR,...,0.0,9.0,17.0,0.0,138.0,2322.0,13.0,NaN,NaN,NaN
2,1980,1,3,CIN,00-0011825,MunoAn00,NaN,Anthony Munoz,True,T,...,0.0,0.0,0.0,0.0,7.0,18.0,4.0,NaN,NaN,NaN
3,1980,1,4,GNB,CLA212454,ClarBr23,bruce-clark-1,Bruce Clark,False,DE,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,1.0,39.5
4,1980,1,5,BAL,DIC442976,DickCu00,curtis-dickey-1,Curtis Dickey,False,RB,...,0.0,937.0,4019.0,32.0,134.0,1577.0,8.0,NaN,NaN,NaN


In [17]:
for i, col in enumerate(draft.columns):
    print(i, col)

print("\nAV columns:", [c for c in draft.columns if "av" in c.lower()])

0 season
1 round
2 pick
3 team
4 gsis_id
5 pfr_player_id
6 cfb_player_id
7 pfr_player_name
8 hof
9 position
10 category
11 side
12 college
13 age
14 to
15 allpro
16 probowls
17 seasons_started
18 w_av
19 car_av
20 dr_av
21 games
22 pass_completions
23 pass_attempts
24 pass_yards
25 pass_tds
26 pass_ints
27 rush_atts
28 rush_yards
29 rush_tds
30 receptions
31 rec_yards
32 rec_tds
33 def_solo_tackles
34 def_ints
35 def_sacks

AV columns: ['w_av', 'car_av', 'dr_av']


In [18]:
cols = [
    "season", "round", "pick", "team", "pfr_player_name", "position",
    "age", "to", "games", "seasons_started", "allpro", "probowls",
    "w_av", "car_av", "dr_av",
]

recent = draft[draft["season"].between(2021, 2026)][cols].copy()
print("shape:", recent.shape)

recent.groupby("season").agg(
    picks=("pick", "size"),
    mean_dr_av=("dr_av", "mean"),
    max_dr_av=("dr_av", "max"),
    missing_dr_av=("dr_av", lambda s: s.isna().sum()),
)

shape: (1551, 15)


,picks,mean_dr_av,max_dr_av,missing_dr_av
season,,,,
2021,259,11.008368,64.0,20
2022,262,10.617284,44.0,19
2023,259,8.097046,40.0,22
2024,257,5.406114,26.0,28
2025,257,2.625000,11.0,33
2026,257,NaN,NaN,257


In [19]:
# 2026 has no outcomes yet — prediction target, not backtest
backtest = recent[recent["season"] <= 2025].copy()

# Missing means "never accumulated value", not "unknown"
backtest["dr_av"] = backtest["dr_av"].fillna(0)

# Standardize within draft class
backtest["dr_av_z"] = backtest.groupby("season")["dr_av"].transform(
    lambda s: (s - s.mean()) / s.std()
)
backtest["dr_av_pct"] = backtest.groupby("season")["dr_av"].rank(pct=True)

backtest.groupby("season")["dr_av_z"].agg(["mean", "std", "min", "max"]).round(3)

,mean,std,min,max
season,,,,
2021,-0.0,1.0,-0.804,4.263
2022,-0.0,1.0,-0.970,3.364
2023,0.0,1.0,-0.900,3.961
2024,0.0,1.0,-0.889,3.909
2025,0.0,1.0,-0.957,3.645


In [20]:
from pathlib import Path

OUT = Path("..") / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

backtest.to_csv(OUT / "draft_outcomes_2021_2025.csv", index=False)

predict_2026 = recent[recent["season"] == 2026].copy()
predict_2026.to_csv(OUT / "draft_2026_picks.csv", index=False)

print("backtest:", backtest.shape)
print("2026:", predict_2026.shape)

backtest: (1294, 17)
2026: (257, 15)


In [21]:
import pandas as pd

outcomes = pd.read_csv("../data/processed/draft_outcomes_2021_2025.csv")
print(sorted(outcomes["team"].unique()))
print("count:", outcomes["team"].nunique())

['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GNB', 'HOU', 'IND', 'JAX', 'KAN', 'LAC', 'LAR', 'LVR', 'MIA', 'MIN', 'NOR', 'NWE', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SFO', 'TAM', 'TEN', 'WAS']
count: 32
